In [ ]:
import os
import sys

try:
    # This will work if running as a .py script
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    # This will be executed if __file__ is not defined (e.g., in a Jupyter Notebook)
    current_script_dir = os.getcwd()

# Navigate up two levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))

# Add the 'GlobalLocal' directory to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root)  # insert at the beginning to prioritize it

import pickle
import pandas as pd
import numpy as np

from src.analysis.decoding.decoding import plot_accuracies_with_multiple_sig_clusters
from src.config.condition_registry import CONDITION_REGISTRY


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)



Qt5Agg backend not available, using default backend
['/hpc/group/coganlab/jz421/GlobalLocal', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python311.zip', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/lib-dynload', '', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages', 'C:/Users/jz421/Desktop/GlobalLocal/IEEG_Pipelines/']
Qt5Agg backend not available, using default backend


load in the master results files

In [3]:
def load_master_results_file(base_path, epochs_root_file, master_results_file):
    with open(os.path.join(base_path, epochs_root_file, master_results_file), "rb") as f:
        master_results = pickle.load(f)
    return master_results

In [16]:
base_path = "/cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/decoding/figs"
save_dir = "/hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots"
epochs_root_file = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

# update these once done running decoding
results_files = {
    'congruency_blockA_results_file': "20260227_110725_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockA_conditions.pkl",
    'congruency_blockB_results_file': "20260227_110725_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockB_conditions.pkl",
    'congruency_blockC_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockC_conditions.pkl",
    'congruency_blockD_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_congruency_blockD_conditions.pkl",
    'switchType_blockA_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockA_conditions.pkl",
    'switchType_blockB_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockB_conditions.pkl",
    'switchType_blockC_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockC_conditions.pkl",
    'switchType_blockD_results_file': "20260227_110825_MASTER_RESULTS_19_subs_all_elecs_LDA_20boots_5splits_5reps_repeat_unit_ev_0.9_stimulus_switchType_blockD_conditions.pkl"
}

results = {}

for results_name, results_filepath in results_files.items():
    results[results_name] = load_master_results_file(base_path, epochs_root_file, results_filepath)


In [35]:
def plot_block_decoding_comparison(
    results_dict,
    block_metadata,
    decoding_type,
    roi_to_plot='lpfc',
    save_dir='.',
    epochs_root_file='',
    ylim=(0.3, 0.9),
    single_column=True,
    show_legend=True,
    filename_suffix='poster_style',
):
    first_key = next(iter(results_dict))
    first_results = results_dict[first_key]
    unit = first_results['metadata']['args']['unit_of_analysis']
    time_points = first_results['metadata']['time_window_centers']
    window_size = first_results['metadata']['args']['window_size']
    step_size = first_results['metadata']['args']['step_size']
    sampling_rate = first_results['metadata']['args']['sampling_rate']
    
    save_directory = os.path.join(save_dir, epochs_root_file, f"{roi_to_plot}_poster_plots")
    os.makedirs(save_directory, exist_ok=True)
    
    accuracies_dict = {}
    colors = {}
    linestyles = {}
    shuffle_accs_list = []
    
    for block_key, meta in block_metadata.items():
        label = meta['label']
        block_results = results_dict[block_key]
        comp_key = next(iter(block_results['stats'].keys()))
        
        accuracies_dict[label] = block_results['stats'][comp_key][roi_to_plot][f'{unit}_true_accs']
        colors[label] = meta['color']
        linestyles[label] = meta['linestyle']
        
        shuffle_accs_list.append(
            block_results['stats'][comp_key][roi_to_plot][f'{unit}_shuffle_accs']
        )
    
    pooled_chance = np.mean(np.stack(shuffle_accs_list, axis=0), axis=0)
    accuracies_dict['Chance'] = pooled_chance
    colors['Chance'] = '#949494'
    linestyles['Chance'] = '--'
    
    ylabel = {
        'congruency': 'Congruency Decoding Accuracy',
        'switchType': 'Switch Type Decoding Accuracy',
    }.get(decoding_type, 'Decoding Accuracy')
    
    plot_accuracies_with_multiple_sig_clusters(
        time_points=time_points,
        accuracies_dict=accuracies_dict,
        significance_clusters_dict={},
        colors=colors,
        linestyles=linestyles,
        ylabel=ylabel,
        ylim=ylim,
        show_chance_level=False,
        show_sig_legend=False,
        comparison_name=f'{decoding_type}_block_comparison',
        roi=roi_to_plot,
        save_dir=save_directory,
        filename_suffix=filename_suffix,
        window_size=window_size,
        step_size=step_size,
        sampling_rate=sampling_rate,
        single_column=single_column,
        show_legend=show_legend,
    )
    print(f"✅ Plot saved in: {save_directory}")

In [36]:
block_metadata = {
    'blockD': {'label': 'Poster A (25I/75S)', 'color': '#FF7E79', 'linestyle': '--'},
    'blockB': {'label': 'Poster B (75I/75S)', 'color': '#DE8F05', 'linestyle': '--'},
    'blockC': {'label': 'Poster C (25I/25S)', 'color': '#FF7E79', 'linestyle': '-'},
    'blockA': {'label': 'Poster D (75I/25S)', 'color': '#DE8F05', 'linestyle': '-'},
}

# Congruency
plot_block_decoding_comparison(
    results_dict={k.replace('congruency_', '').replace('_results_file', ''): v 
                  for k, v in results.items() if 'congruency' in k},
    block_metadata=block_metadata,
    decoding_type='congruency',
    roi_to_plot='lpfc',
    save_dir=save_dir,
    epochs_root_file=epochs_root_file,
)

# Switch type
plot_block_decoding_comparison(
    results_dict={k.replace('switchType_', '').replace('_results_file', ''): v 
                  for k, v in results.items() if 'switchType' in k},
    block_metadata=block_metadata,
    decoding_type='switchType',
    roi_to_plot='lpfc',
    save_dir=save_dir,
    epochs_root_file=epochs_root_file,
)

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved Nature-style plot with multiple significance clusters to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots/congruency_block_comparison_lpfc_poster_style.pdf
✅ Plot saved in: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit/lpfc_poster_plots
Saved Nature-style plot with multiple significance clusters to: /hpc/home/jz421/coganlab/jz421/GlobalLocal/dcc_scripts/decoding/block_by_block_cns26_poster_plots/Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLengt